# Fitting the models and additional control of `fit_gam` in tradeSeq-python

Python port of the R vignette `vignettes/fitGAM.Rmd`. The companion
`tradeSeq.ipynb` walks through the analysis workflow; this notebook
focuses on the **mechanics of fitting** — knot selection, covariates,
parallelism, gene subsetting, list-mode output, and convergence.


# Introduction

`tradeSeq` fits gene-wise negative-binomial generalized additive models
along trajectories and performs inference through contrasts of fitted
GAM parameters. This Python notebook follows the R `fitGAM` vignette but
uses only AnnData slots; the R container and trajectory accessor calls
collapse to direct reads from the bundled h5ad fixture.


## Installation

```bash
pip install tradeSeq-python
```


## Load data

In [ ]:
import numpy as np
import pandas as pd

import tradeseq as ts

adata = ts.load_paul15()
adata


We find two lineages for this dataset. The trajectory can be visualized
with `plot_gene_count`, using cell type labels to colour the cells:


In [ ]:
ts.plot_gene_count(adata, clusters_key='celltype', title='Colored by cell type')


# Choosing K: a deeper dive into the output from `evaluate_k`

`evaluate_k` refits an NB-GAM at each candidate knot count, recording
the per-gene AIC. The R version produces a 4-panel diagnostic figure
via `par(mfrow = c(1, 4))`; the Python version composes the same four
panels with `patchwork.wrap_plots(..., nrow=1)`.


### Downstream of any trajectory-inference method

In [ ]:
aic_mat = ts.evaluate_k(
    adata,
    k_range=range(3, 8),  # try k = 3 ... 7
    n_genes=100,
    plot=False,
    random_state=5,
)
aic_mat.shape


In [ ]:
diag_plot = ts.plot_evaluatek_results(aic_mat, k_range=range(3, 8))
diag_plot


The R vignette interprets these panels as follows (the same
interpretation applies here):

1. **Boxplot** — per-gene AIC deviations from the average across `k`.
2. **Mean AIC** — should plateau around an optimal `k`.
3. **Mean relative AIC** — the elbow indicates the optimal `k`.
4. **Optimal-k histogram** — distribution of per-gene optimal knot
   counts (the mode is the recommended choice).


# Fit additive models

The full fit. With 240 genes × 2660 cells this takes a couple of
minutes; for the notebook we restrict to 30 genes.


In [ ]:
# For reproducibility.
rng = np.random.default_rng(7)
probs = adata.obsm['cell_weights'] / adata.obsm['cell_weights'].sum(axis=1, keepdims=True)
w_samp = np.zeros_like(adata.obsm['cell_weights'], dtype=np.int64)
for i in range(adata.n_obs):
    w_samp[i] = rng.multinomial(1, probs[i])

a = adata[:, :30].copy()
ts.fit_gam(a, n_knots=6, verbose=False, _w_samp=w_samp)
a


Inspect the convergence flag — R uses `try(withCallingHandlers(...))`
to catch BOTH errors AND warnings; Python mirrors that via
`warnings.catch_warnings(record=True)`. Either kind of issue marks the
gene as `converged=False`.


In [ ]:
a.var['tradeseq_converged'].value_counts()


## Adding covariates to the model

You can include extra fixed effects via the `U` argument (the design
matrix of fixed effects, `(n_cells, k)`). For instance, to adjust for
a hypothetical batch effect:


In [ ]:
batch_indicator = np.random.default_rng(0).integers(0, 2, size=a.n_obs).reshape(-1, 1).astype(float)
ts.fit_gam(
    a,
    U=batch_indicator,  # (n_cells, 1)
    n_knots=6,
    verbose=False,
    _w_samp=w_samp,
)


## Parallel computing

Set `parallel=True` (and `n_jobs > 1`) to dispatch per-gene fits via
joblib (R uses `BiocParallel::bplapply`):


In [ ]:
ts.fit_gam(
    a, parallel=True, n_jobs=2, n_knots=6,
    verbose=False, _w_samp=w_samp,
)


## Fitting only a subset of genes

Pass `genes=` (a list of gene names or integer indices). Library size
and TMM normalisation use ALL genes — only the smoothers are
restricted.


In [ ]:
ts.fit_gam(
    a, genes=list(a.var_names[:10]),
    n_knots=6, verbose=False, _w_samp=w_samp,
)
a.var['tradeseq_converged'].head(15)


## Zero inflation

If you have ZINB observation weights (e.g. from a zero-inflation
decoupling step), pass them via `sample_weights` (shape `(n_genes,
n_cells)`). The weights enter the GLM as observation weights, exactly
as in the R source's `weights = weights[teller, ]` at
`fitGAM.R:261`.


In [ ]:
# Stub example — weights all 1.0; in production use real ZINB output.
weights = np.ones((a.n_vars, a.n_obs))
ts.fit_gam(
    a, sample_weights=weights, n_knots=6,
    verbose=False, _w_samp=w_samp,
)


## Convergence issues on small or zero-inflated datasets

Genes can fail to converge due to extreme sparsity. Inspect the
`tradeseq_converged` flag and consider raising knots, applying ZINB
weights, or excluding the offending genes.


In [ ]:
failed = a.var.index[~a.var['tradeseq_converged']].tolist()
print(f'{len(failed)} of {a.n_vars} genes failed to converge.')


## tradeSeq list output

Setting `sce=False` returns a `dict[gene_name, FittedGam]` instead of
mutating the AnnData. The `FittedGam` dataclass exposes the minimal
mgcv-`gam` attributes that the downstream exports consume:
`coef_`, `cov_`, `lpmatrix_`, `dm_`, `summary_s_table`, `alpha_`,
`converged_`. This mode is required by `get_smoother_pvalues` and
`get_smoother_test_stats`.


In [ ]:
fits = ts.fit_gam(
    a, n_knots=6, verbose=False, _w_samp=w_samp, sce=False,
)
type(fits), len(fits)


In [ ]:
first_gene, first_fit = next(iter(fits.items()))
print('gene:', first_gene)
print('coef_ shape:', first_fit.coef_.shape)
print('cov_ shape:', first_fit.cov_.shape)
print('summary_s_table:')
first_fit.summary_s_table


`get_smoother_pvalues` returns per-(gene, smoother) p-values:

In [ ]:
ts.get_smoother_pvalues(fits).head()


And `get_smoother_test_stats` the matching chi-squared statistics:

In [ ]:
ts.get_smoother_test_stats(fits).head()


# Wrap-up

The full surface for `fit_gam` is captured by these knobs:

- `layer`, `pseudotime_key`, `weights_key` — AnnData slot names
- `U` — fixed-effect design matrix
- `conditions_key` — name of an `obs` column for conditions
- `genes` — fit a subset (defaults to all)
- `sample_weights` — ZINB observation weights
- `offset` — explicit log-offset (overrides TMM)
- `n_knots`, `family` (`"nb"` or `"gaussian"`)
- `parallel`, `n_jobs`, `verbose`
- `sce` — AnnData vs `dict[str, FittedGam]` output
- `key_added` — namespace prefix (default `"tradeseq"`)
- `copy` — mutate in place vs return a copy
- `_w_samp` — private validation hook for R-side multinomial draws

See `tradeSeq.ipynb` for the full analysis workflow.


# Session

In [ ]:
import platform, sys
print('python:', sys.version.split()[0])
print('platform:', platform.platform())
print('tradeseq:', ts.__version__)


# References

This notebook is the Python/scverse counterpart of
`tradeSeq/vignettes/fitGAM.Rmd`; see the original vignette bibliography
for the statistical model and package citations.
